In [3]:
import yfinance as yf
import pandas as pd
import sqlite3

# ── 1. PULL BENCHMARK DATA ────────────────────────────────────
benchmarks = {
    'FTSE 100':       '^FTSE',
    'S&P 500':        '^GSPC',
    'FTSE All World': 'VWRL.L'
}

benchmark_prices = {}
for name, ticker in benchmarks.items():
    data = yf.download(ticker, period='3y', interval='1mo', progress=False, auto_adjust=True)
    benchmark_prices[name] = data['Close']
    print(f"Downloaded {name}: {len(data)} months")

benchmarks_df = pd.concat(benchmark_prices, axis=1)
benchmarks_df.columns = benchmarks_df.columns.droplevel(1)

# ── 2. PULL ACTIVE TRUST DATA ─────────────────────────────────
active_trusts = {
    'City of London':                    'CTY.L',
    'JPMorgan Global Growth & Income':   'JGGI.L',
    'F&C Investment Trust':              'FCIT.L',
    'Scottish Mortgage':                 'SMT.L',
    'Murray International':              'MYI.L',
    'Monks Investment Trust':            'MNKS.L',
    'Bankers Investment Trust':          'BNKR.L',
    'Henderson Far East Income':         'HFEL.L',
    'Scottish American':                 'SAIN.L',
    'Merchants Trust':                   'MRCH.L',
    'Lowland Investment Trust':          'LWI.L',
    'Law Debenture':                     'LWDB.L',
    'Temple Bar':                        'TMPL.L',
    'BlackRock World Mining':            'BRWM.L',
    'JPMorgan European Growth & Income': 'JEGI.L',
    'TwentyFour Income Fund':            'TFIF.L',
    'International Public Partnerships': 'INPP.L',
    'Personal Assets Trust':             'PNL.L'
}

active_prices = {}
for name, ticker in active_trusts.items():
    data = yf.download(ticker, period='3y', interval='1mo', progress=False, auto_adjust=True)
    if len(data) > 0:
        active_prices[name] = data['Close']
        print(f"✅ {name}: {len(data)} months")
    else:
        print(f"❌ {name}: no data")

active_df = pd.concat(active_prices, axis=1)
active_df.columns = active_df.columns.droplevel(1)

# ── 3. MERGE AND REBASE TO 1 ──────────────────────────────────
all_data = pd.concat([benchmarks_df, active_df], axis=1).dropna()

# Rebase to 1 (cleaner than 100)
rebased = (all_data / all_data.iloc[0]).round(4)

# Add Active Fund Average
active_cols = list(active_prices.keys())
rebased['Active Fund Average'] = rebased[active_cols].mean(axis=1).round(4)

# ── 4. CONVERT TO LONG FORMAT ─────────────────────────────────
rebased.index.name = 'Date'
rebased = rebased.reset_index()

# Melt to long format - Date, Fund, Value only
long_df = rebased.melt(id_vars='Date', var_name='Fund', value_name='Value')

# Sort by Date then Fund
long_df['Date'] = pd.to_datetime(long_df['Date']).dt.strftime('%Y-%m-%d')
long_df = long_df.sort_values(['Date', 'Fund']).reset_index(drop=True)

print(f"\nShape: {long_df.shape}")
print(long_df.head(10))

# ── 5. SAVE TO CSV AND SQLITE ─────────────────────────────────
long_df.to_csv(r'D:\Data\active vs passive\active_vs_passive_long.csv', index=False)

conn = sqlite3.connect(r'D:\Data\active vs passive\active_vs_passive.db')
long_df.to_sql('performance_long', conn, if_exists='replace', index=False)
conn.close()

print("\nSaved to CSV and SQLite ✅")

Downloaded FTSE 100: 36 months
Downloaded S&P 500: 36 months
Downloaded FTSE All World: 36 months
✅ City of London: 36 months
✅ JPMorgan Global Growth & Income: 36 months
✅ F&C Investment Trust: 36 months
✅ Scottish Mortgage: 36 months
✅ Murray International: 36 months
✅ Monks Investment Trust: 36 months
✅ Bankers Investment Trust: 36 months
✅ Henderson Far East Income: 36 months
✅ Scottish American: 36 months
✅ Merchants Trust: 36 months
✅ Lowland Investment Trust: 36 months
✅ Law Debenture: 36 months
✅ Temple Bar: 36 months
✅ BlackRock World Mining: 36 months
✅ JPMorgan European Growth & Income: 36 months
✅ TwentyFour Income Fund: 36 months
✅ International Public Partnerships: 36 months
✅ Personal Assets Trust: 36 months

Shape: (792, 3)
         Date                               Fund  Value
0  2023-05-01                Active Fund Average    1.0
1  2023-05-01           Bankers Investment Trust    1.0
2  2023-05-01             BlackRock World Mining    1.0
3  2023-05-01             